In [1]:
##load dataset
import pandas as pd
df = pd.read_csv("dataset.csv")
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [20]:
#remove useless columns ,so we are removing udi and product id because they are just IDs (no meaning for prediction)
# df = df.drop(['UDI', 'Product ID'], axis=1)
df = df.drop(columns=['UDI', 'Product ID'], errors='ignore')
df.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,False,True
1,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,True,False
2,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,True,False
3,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,True,False
4,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,True,False


In [5]:
#to know the more about the dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Type                     10000 non-null  str    
 1   Air temperature [K]      10000 non-null  float64
 2   Process temperature [K]  10000 non-null  float64
 3   Rotational speed [rpm]   10000 non-null  int64  
 4   Torque [Nm]              10000 non-null  float64
 5   Tool wear [min]          10000 non-null  int64  
 6   Machine failure          10000 non-null  int64  
 7   TWF                      10000 non-null  int64  
 8   HDF                      10000 non-null  int64  
 9   PWF                      10000 non-null  int64  
 10  OSF                      10000 non-null  int64  
 11  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(8), str(1)
memory usage: 937.6 KB


In [6]:
#Convert categorical (text) columns into numerical here type
df = pd.get_dummies(df, drop_first=True)

In [7]:
# Separate features (input) and target (output)
# X -= all sensor data (temperature, torque, etc.)
# y -- result (0 = no failure, 1 = failure)
X = df.drop('Machine failure', axis=1)
y = df['Machine failure']

In [8]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
from imblearn.over_sampling import SMOTE
# Apply SMOTE to balance the dataset
smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(y_train.value_counts())      # before
print(y_train_sm.value_counts())   # after

Machine failure
0    7722
1     278
Name: count, dtype: int64
Machine failure
0    7722
1    7722
Name: count, dtype: int64


In [10]:
from sklearn.ensemble import RandomForestClassifier
# Create and train the model
model = RandomForestClassifier()
model.fit(X_train_sm, y_train_sm)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [11]:
from sklearn.metrics import accuracy_score

# Make predictions
y_pred = model.predict(X_test)
# # Get probability of failure (class 1)
# y_prob = model.predict_proba(X_test)[:, 1]

# # Apply custom threshold (instead of default 0.5)
# y_pred = (y_prob > 0.3).astype(int)

# Check accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9835


In [12]:
import joblib
#see the trained model
joblib.dump(model, "model.pkl")

['model.pkl']

In [25]:
from sklearn.metrics import classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9835
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      1939
           1       0.66      0.97      0.78        61

    accuracy                           0.98      2000
   macro avg       0.83      0.98      0.89      2000
weighted avg       0.99      0.98      0.99      2000



In [14]:
# Example new machine data (random sample)
sample = X_test.iloc[0].values.reshape(1, -1)

prediction = model.predict(sample)

if prediction[0] == 1:
    print("⚠️ Machine Failure Risk!")
else:
    print("✅ Machine is Safe")

✅ Machine is Safe


e:\Vidhi\2nd year\4th sem\ai\project2\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [15]:
import joblib
joblib.dump(model, "model.pkl")

['model.pkl']

## Phase 1: Neural Network (MLP)

In [16]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler

# Neural networks require scaled features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)

# Build MLP Model
mlp_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid') # Binary classification
])

mlp_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train MLP
print("Training MLP Model...")
history = mlp_model.fit(X_train_scaled, y_train_sm, epochs=20, batch_size=32, validation_split=0.2, verbose=1)

# Evaluate
mlp_loss, mlp_acc = mlp_model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"MLP Test Accuracy: {mlp_acc:.4f}")

# Save MLP model
mlp_model.save('mlp_model.keras')


Training MLP Model...
Epoch 1/20


e:\Vidhi\2nd year\4th sem\ai\project2\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


387/387 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9236 - loss: 0.2275 - val_accuracy: 0.9443 - val_loss: 0.1428
Epoch 2/20
387/387 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9636 - loss: 0.1040 - val_accuracy: 0.9246 - val_loss: 0.1858
Epoch 3/20
387/387 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9653 - loss: 0.0957 - val_accuracy: 0.9314 - val_loss: 0.1566
Epoch 4/20
387/387 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9667 - loss: 0.0920 - val_accuracy: 0.9547 - val_loss: 0.1353
Epoch 5/20
387/387 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9679 - loss: 0.0848 - val_accuracy: 0.9440 - val_loss: 0.1351
Epoch 6/20
387/387 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9701 - loss: 0.0855 - val_accuracy: 0.9375 - val_loss: 0.1468
Epoch 7/20
387/387 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9696 - loss: 0.0818 - val_accuracy: 0.9602 - val_loss: 0.1142
Epoch 8/20
387/387 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9700 - loss: 0.0797 - val_accuracy: 0.9582 - val_

## Phase 2: Anomaly Detection (Autoencoder)

In [17]:

import numpy as np

# Filter only "normal" data for training the autoencoder (Machine failure == 0)
X_train_normal = X_train[y_train == 0]
X_train_normal_scaled = scaler.transform(X_train_normal) # use the same scaler

# Build Autoencoder
input_dim = X_train_normal_scaled.shape[1]
autoencoder = Sequential([
    # Encoder
    Dense(16, activation='relu', input_shape=(input_dim,)),
    Dense(8, activation='relu'),
    # Decoder
    Dense(16, activation='relu'),
    Dense(input_dim, activation='linear')
])

autoencoder.compile(optimizer='adam', loss='mse')

# Train Autoencoder
print("Training Autoencoder on normal data...")
autoencoder.fit(X_train_normal_scaled, X_train_normal_scaled, epochs=20, batch_size=32, validation_split=0.2, verbose=1)

# Save Autoencoder
autoencoder.save('autoencoder.keras')

# Predict and calculate reconstruction error on test set
X_test_recon = autoencoder.predict(X_test_scaled)
mse = np.mean(np.power(X_test_scaled - X_test_recon, 2), axis=1)

# Define an anomaly threshold (e.g., 95th percentile of training normal errors)
train_recon = autoencoder.predict(X_train_normal_scaled)
train_mse = np.mean(np.power(X_train_normal_scaled - train_recon, 2), axis=1)
threshold = np.percentile(train_mse, 95)
print(f"Anomaly Detection Threshold (MSE): {threshold:.4f}")

# Flag anomalies
anomalies = mse > threshold
print(f"Number of anomalies detected in test set: {np.sum(anomalies)} out of {len(anomalies)}")


Training Autoencoder on normal data...
Epoch 1/20


e:\Vidhi\2nd year\4th sem\ai\project2\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


194/194 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5462 - val_loss: 0.3263
Epoch 2/20
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2495 - val_loss: 0.1539
Epoch 3/20
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1149 - val_loss: 0.0664
Epoch 4/20
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0500 - val_loss: 0.0418
Epoch 5/20
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0377 - val_loss: 0.0349
Epoch 6/20
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0325 - val_loss: 0.0303
Epoch 7/20
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0270 - val_loss: 0.0241
Epoch 8/20
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0217 - val_loss: 0.0199
Epoch 9/20
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0187 - val_loss: 0.0190
Epoch 10/20
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0172 - val_loss: 0.0165
Epoch 11/20
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0152 - val_loss: 0.0144
Epoch 12/20
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.

## Phase 3: Sequence Modeling (LSTM) for Advance Prediction
*Note: Since the dataset is tabular, we synthesize a sliding window to simulate time-series data.*

In [18]:

# Synthesize sequential data
def create_sequences(X, y, time_steps=5):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X.iloc[i:(i + time_steps)].values)
        ys.append(y.iloc[i + time_steps])
    return np.array(Xs), np.array(ys)

time_steps = 5
X_seq, y_seq = create_sequences(X, y, time_steps)

# Split sequential data
from sklearn.model_selection import train_test_split
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42, shuffle=False) # Important: shuffle=False for time series!

# Scale sequential data (flatten, scale, reshape back)
num_samples_train, seq_len, num_features = X_train_seq.shape
X_train_seq_flat = X_train_seq.reshape(-1, num_features)
X_train_seq_scaled_flat = scaler.fit_transform(X_train_seq_flat)
X_train_seq_scaled = X_train_seq_scaled_flat.reshape(num_samples_train, seq_len, num_features)

num_samples_test = X_test_seq.shape[0]
X_test_seq_flat = X_test_seq.reshape(-1, num_features)
X_test_seq_scaled_flat = scaler.transform(X_test_seq_flat)
X_test_seq_scaled = X_test_seq_scaled_flat.reshape(num_samples_test, seq_len, num_features)


from tensorflow.keras.layers import LSTM

# Build LSTM
lstm_model = Sequential([
    LSTM(32, activation='relu', input_shape=(seq_len, num_features)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train LSTM
print("Training LSTM Model...")
# Note: dealing with class imbalance is harder here, but we will train it simply for demonstration
lstm_model.fit(X_train_seq_scaled, y_train_seq, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Evaluate
lstm_loss, lstm_acc = lstm_model.evaluate(X_test_seq_scaled, y_test_seq, verbose=0)
print(f"LSTM Test Accuracy: {lstm_acc:.4f}")

# Save LSTM
lstm_model.save('lstm_model.keras')


Training LSTM Model...
Epoch 1/10


e:\Vidhi\2nd year\4th sem\ai\project2\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9447 - loss: 0.2456 - val_accuracy: 0.9781 - val_loss: 0.1099
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9586 - loss: 0.1677 - val_accuracy: 0.9781 - val_loss: 0.1065
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9586 - loss: 0.1621 - val_accuracy: 0.9781 - val_loss: 0.1048
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9586 - loss: 0.1603 - val_accuracy: 0.9781 - val_loss: 0.1096
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9586 - loss: 0.1572 - val_accuracy: 0.9781 - val_loss: 0.1067
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9586 - loss: 0.1568 - val_accuracy: 0.9781 - val_loss: 0.1054
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9586 - loss: 0.1562 - val_accuracy: 0.9781 - val_loss: 0.1051
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9584 - loss: 0.1537 - val_accuracy: 0.9781 - val_